# Figure 1
Load pre-computed data and generate the figure.

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

# Load saved figure data
save_dir = "figure_data/figure_1/"

df_fire = pd.read_csv(save_dir + "fire_timeseries.csv")
burned_forest_tseries = xr.DataArray(
    df_fire["burned_forest"].values, coords={"year": df_fire["year"].values}, dims=["year"]
)
forest_area = xr.DataArray(
    df_fire["forest_area"].values, coords={"year": df_fire["year"].values}, dims=["year"]
)

df_fia = pd.read_csv(save_dir + "fia_timeseries.csv")
fia_frac_plots = xr.DataArray(
    df_fia["fia_frac_plots"].values, coords={"year": df_fia["year"].values}, dims=["year"]
)

with open(save_dir + "target_crs.txt") as f:
    crs = f.read()

mapdata_clipped = xr.open_dataset(save_dir + "mapdata_clipped.nc")["mapdata_clipped"].rio.write_crs(
    crs
)
is_forest_clipped = xr.open_dataset(save_dir + "is_forest_clipped.nc")[
    "is_forest_clipped"
].rio.write_crs(crs)

western_states = maps.SHP_WESTERN.to_crs(crs)

# Recreate colormap
cmap = mcolors.LinearSegmentedColormap.from_list("my_cmap", ["#FBBBA8", "#690F0F"])
n_colors = 5
discrete_cmap = mcolors.ListedColormap(cmap(np.linspace(0, 1, n_colors)))

In [ ]:
fig = plt.figure(figsize=(24, 10))
font_settings = {
    "font.size": 24,
    "axes.titlesize": 28,
    "axes.labelsize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
}
plt.rcParams.update(font_settings)


# Create axes with width ratios: left panel smaller, right panel bigger
gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1], wspace=0.05)
axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]

# Left plot - time series
ax0 = axes[0]
yvals = burned_forest_tseries * 100 / forest_area
ax0.plot(yvals["year"], yvals, "-", color="#B12121", linewidth=6, markersize=15, label="MTBS")
ax0.plot(
    yvals["year"],
    yvals.rolling(year=10, center=True).mean(),
    "--",
    color="#B12121",
    linewidth=6,
    label="MTBS (centered 10-year rolling mean)",
)
ax0.axhline(y=0, linestyle=":", color="k")
ax0.set_ylabel("Burned area (percent of forest area per year)", labelpad=10)
ax0.set_xlabel("Year", labelpad=10)
ax0.spines[["right", "top"]].set_visible(True)
ax0.set_title("Forest area burned annually", pad=15)
ax0.plot(
    fia_frac_plots["year"],
    fia_frac_plots.where(fia_frac_plots["year"] >= 2000),
    color="#E55713",
    linewidth=6,
    label="FIA-derived",
)
ax0.legend()
ax0.text(
    -0.08,
    1.05,
    "a",
    transform=ax0.transAxes,
    fontsize=font_settings["axes.titlesize"] * 1.25,
    fontweight="bold",
    va="top",
)
ax0.tick_params(axis="y", which="both", left=True, right=True, direction="in")
ax0.tick_params(axis="x", which="both", top=True, bottom=True, direction="in")
ax0.legend(frameon=False)

# Right plot - maps
ax1 = axes[1]
maps.plot_map(
    is_forest_clipped,
    shp=western_states,
    latlon=False,
    cbar_label="Percent Forest",
    cmap=plt.cm.binary,
    add_colorbar=False,
    clims=[0, 600],
    savefig=None,
    ax=ax1,
)
maps.plot_map(
    mapdata_clipped,
    shp=western_states,
    latlon=False,
    title="Year of most recent fire",
    cbar_label="Year of most recent fire",
    cmap=discrete_cmap,
    savefig=None,
    clims=[2000, 2025],
    add_colorbar=True,
    ax=ax1,
)
ax1.set_title("Year of most recent fire", pad=15)

plt.rcParams.update(font_settings)

# The colorbar is on a separate axes appended to the figure
for ax in fig.axes:
    if ax != ax0 and ax != ax1:  # it's the colorbar axes
        ax.yaxis.label.set_fontsize(font_settings["axes.titlesize"])
        ax.tick_params(labelsize=font_settings["xtick.labelsize"])
        ax.set_ylabel("Year of most recent fire", labelpad=10)

ax1.text(
    0,
    1.05,
    "b",
    transform=ax1.transAxes,
    fontsize=font_settings["axes.titlesize"] * 1.25,
    fontweight="bold",
    va="top",
)
fig.subplots_adjust(left=0.05, right=0.92, top=0.92, bottom=0.08, wspace=-0.1)
fig.savefig(dir_info.dir_figures + "Figure1.jpg", bbox_inches="tight", dpi=300)
# fig.savefig(dir_info.dir_figures + "Figure1.pdf", bbox_inches="tight", dpi=300)